# Credit Card Transaction Analysis & Fraud Detection
## Stage 4: Association Rule Mining (Apriori & FP-Growth)

In this notebook, we transform transaction attributes (category, discretized amount, discretized hour, gender, weekend flag) into itemsets to extract frequent patterns and association rules associated with credit card fraud.

We compare:
- **Apriori Algorithm**
- **FP-Growth Algorithm**

And evaluate rules using **Support**, **Confidence**, and **Lift**.

### Step 1: Load and Prepare Basket Data

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

clean_train_path = os.path.join("..", "data", "processed", "fraudTrain_clean.csv")
df = pd.read_csv(clean_train_path)

def discretize_hour(hour):
    if 22 <= hour or hour < 5:
        return "hour=Late_Night"
    elif 5 <= hour < 12:
        return "hour=Morning"
    elif 12 <= hour < 17:
        return "hour=Afternoon"
    else:
        return "hour=Evening"

# Sample fraud transactions with a balanced contrast of genuine
fraud_df = df[df["is_fraud"] == 1]
genuine_df = df[df["is_fraud"] == 0].sample(n=min(len(fraud_df) * 3, 30000), random_state=42)
sample_df = pd.concat([fraud_df, genuine_df]).sample(frac=1, random_state=42).reset_index(drop=True)

transactions = []
for _, row in sample_df.iterrows():
    items = [
        f"cat={row['category']}",
        f"amt={row['amount_category']}",
        discretize_hour(row["transaction_hour"]),
        f"gender={row['gender']}",
        "weekend=Yes" if row["is_weekend"] == 1 else "weekend=No",
        "fraud=Yes" if row["is_fraud"] == 1 else "fraud=No"
    ]
    transactions.append(items)

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
print(f"Encoded matrix shape: {df_encoded.shape}")

### Step 2: Compare Apriori vs FP-Growth Runtimes

In [ ]:
min_support = 0.05

t0 = time.time()
apriori_itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
time_apriori = time.time() - t0

t0 = time.time()
fpgrowth_itemsets = fpgrowth(df_encoded, min_support=min_support, use_colnames=True)
time_fpgrowth = time.time() - t0

print(f"Apriori: {time_apriori:.3f}s ({len(apriori_itemsets)} itemsets)")
print(f"FP-Growth: {time_fpgrowth:.3f}s ({len(fpgrowth_itemsets)} itemsets)")

plt.figure(figsize=(6, 4))
plt.bar(["Apriori", "FP-Growth"], [time_apriori, time_fpgrowth], color=["#e74c3c", "#2ecc71"])
plt.title("Execution Time (seconds)")
plt.ylabel("Time (s)")
plt.show()

### Step 3: Extract Association Rules (Support, Confidence, Lift)

In [ ]:
rules = association_rules(fpgrowth_itemsets, metric="lift", min_threshold=1.2)
rules["antecedents_str"] = rules["antecedents"].apply(lambda s: ", ".join(list(s)))
rules["consequents_str"] = rules["consequents"].apply(lambda s: ", ".join(list(s)))

fraud_rules = rules[rules["consequents_str"].str.contains("fraud=Yes")].sort_values(by="lift", ascending=False)
print(f"Top rules predicting fraud=Yes:")
fraud_rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]].head(10)